# 第 1 周：从张量到服饰分类

这是配合 src/ 的学习笔记。先运行完整训练，再从上到下执行本笔记；小批量演示使用独立模型，不会覆盖已保存模型。

## 1. 准备环境

在项目虚拟环境中运行 Jupyter。这里兼容从项目根目录或 notebooks 目录启动。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), '请从项目根目录或 notebooks 目录启动'
sys.path.insert(0, str(ROOT))
import torch
from src.train import seed_everything
seed_everything()
torch.set_num_threads(4)
print('PyTorch:', torch.__version__)

## 2. 张量的形状和基本操作

张量可以理解为带形状和类型的多维数组。`reshape` 改变形状；`sum` 求和；`@` 是矩阵乘法，`*` 是逐元素乘法。

In [ ]:
x = torch.arange(6, dtype=torch.float32).reshape(2, 3)
print('x =', x)
print('shape / dtype:', x.shape, x.dtype)
print('第一行:', x[0])
print('逐元素乘 2:', x * 2)
print('每行之和:', x.sum(dim=1))
print('矩阵乘法:', x @ x.T)

## 3. Dataset 和 DataLoader

Dataset 按索引返回一张图片和标签；DataLoader 打乱、分批返回数据。训练和验证来自固定种子划分的原始训练集，测试使用官方独立测试集。

In [ ]:
from src.dataset import make_loaders
train_loader, val_loader, test_loader = make_loaders()
images, labels = next(iter(train_loader))
print('样本数:', len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset))
print('图片:', images.shape, images.dtype)
print('标签:', labels.shape, labels.dtype)
print('展平后:', images.flatten(1).shape)
assert images.shape == (128, 1, 28, 28)
assert set(train_loader.dataset.indices).isdisjoint(val_loader.dataset.indices)

## 4. 一个 batch 的前向、反向和更新

模型输出 `[128,10]` 的原始分数 logits，不预先做 softmax。交叉熵内部会处理这些分数。`backward()` 产生梯度，`step()` 才改变参数。

In [ ]:
from src.model import MLP
model = MLP()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()
weight = next(model.parameters())
before = weight.detach().clone()
optimizer.zero_grad()
logits = model(images)
loss = criterion(logits, labels)
loss.backward()
print('logits:', logits.shape, 'loss:', loss.item())
print('梯度范数:', weight.grad.norm().item())
assert torch.equal(before, weight)
optimizer.step()
print('参数变化总量:', (weight - before).abs().sum().item())
assert not torch.equal(before, weight)

## 5. 检查真实实验

先在终端运行 `python -m src.train`。以下只读取指标和最佳模型，不重新训练，也不使用测试集调参。

In [ ]:
import json
metrics = json.loads((ROOT / 'outputs/mlp/metrics.json').read_text(encoding='utf-8'))
for key in ['model', 'epochs', 'train_size', 'val_size', 'test_size', 'best_epoch', 'test_loss', 'test_accuracy']:
    print(key, ':', metrics[key])
checkpoint = torch.load(ROOT / 'outputs/mlp/model_best.pt', map_location='cpu', weights_only=True)
trained = MLP()
trained.load_state_dict(checkpoint['model_state'])
trained.eval()
with torch.no_grad():
    predicted = trained(images).argmax(1)
print('一批训练图片的前 10 个预测:', predicted[:10].tolist())

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'outputs/mlp/figures/training_curves.png')))
display(Image(filename=str(ROOT / 'outputs/mlp/predictions/correct.png')))
display(Image(filename=str(ROOT / 'outputs/mlp/predictions/incorrect.png')))

## 6. 下一步

对照 README 和博客解释曲线。尝试先增加 epoch，再单独比较 CNN；一次只改变一个条件，用验证集选择设置。错误图只是测试顺序中的前 8 个错误，不能代表全部错误分布。